# Satellite Power Prediction Using Machine Learning

## 3.1 Introduction

This notebook develops and evaluates machine-learning models for short-term satellite electrical power prediction as part of the **Dream in Orbit — An AI That Simulates a Satellite** project.

The objective is to predict the future value of satellite power variable `NPWD2721` approximately **288 seconds (4.8 minutes) ahead**, using 16 input features derived from the historical behavior of `NPWD2721` and `NPWD2802`.

The analysis uses the model-ready datasets generated in Notebook 02. All preprocessing, feature engineering, chronological data splitting, and feature scaling have already been completed and will not be repeated.

### Modeling objectives

1. Establish a baseline for satellite power prediction.
2. Train and evaluate suitable machine-learning models.
3. Compare prediction performance using appropriate regression metrics.
4. Select a model for integration into the satellite power simulation.
5. Save the selected model and its evaluation results for reuse.

### Data organization

The model-ready datasets are stored in `data/processed/` and include separate training, validation, and testing datasets for `annee1` and `annee2`.

The training datasets will be used to fit models, validation datasets for model selection, and test datasets for final evaluation.

## 3.2 — Model Development Setup

### 3.2.1 — Import Required Libraries

This section imports the libraries required for loading processed satellite datasets, developing machine-learning models, evaluating prediction performance, and saving trained models.

In [2]:
# Section 3.2.1 — Import Required Libraries

from pathlib import Path
import numpy as np
import pandas as pd
import joblib

# Machine-learning models
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor

# Model evaluation
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# Visualization
import matplotlib.pyplot as plt

# Reproducibility
RANDOM_STATE = 42

print("All required libraries imported successfully.")

All required libraries imported successfully.


### 3.2.2 — Load Model-Ready Satellite Datasets

This section loads the training, validation, and testing datasets generated in Notebook 02 for both satellite observation years.

The datasets are loaded directly from the saved `.npz` files, preserving the previously completed preprocessing and chronological data splits.

In [3]:
# Section 3.2.2 — Load Model-Ready Satellite Datasets

# Locate the processed dataset directory
project_root = Path.cwd().resolve()
data_dir = project_root / "data" / "processed"

assert data_dir.is_dir(), (
    f"Processed dataset directory not found: {data_dir}"
)

# Load datasets for both observation years
model_data = {}

for year in ("annee1", "annee2"):
    model_data[year] = {}

    for split in ("train", "validation", "test"):
        file_path = data_dir / f"{year}_{split}.npz"

        assert file_path.is_file(), f"Missing dataset: {file_path}"

        with np.load(file_path) as saved:
            X = saved["X"]
            y = saved["y"]

        assert X.shape[0] == y.shape[0]
        assert X.shape[1] == 16

        model_data[year][split] = {
            "X": X,
            "y": y
        }

        print(
            f"{year} | {split:10s} | "
            f"X: {X.shape} | y: {y.shape}"
        )

print("\nAll model-ready satellite datasets loaded successfully.")

annee1 | train      | X: (1277635, 16) | y: (1277635,)
annee1 | validation | X: (273772, 16) | y: (273772,)
annee1 | test       | X: (273781, 16) | y: (273781,)
annee2 | train      | X: (1226455, 16) | y: (1226455,)
annee2 | validation | X: (262805, 16) | y: (262805,)
annee2 | test       | X: (262814, 16) | y: (262814,)

All model-ready satellite datasets loaded successfully.


### 3.2.3 — Load and Verify Feature Metadata

This section loads the feature metadata and fitted scalers saved during preprocessing.

The metadata ensures consistency between the training datasets and the original feature engineering configuration. The fitted scalers are retained for future predictions without refitting them.

In [4]:
# Section 3.2.3 — Load and Verify Feature Metadata

# Load saved preprocessing artifacts
feature_metadata = joblib.load(
    data_dir / "feature_metadata.joblib"
)

feature_scalers = joblib.load(
    data_dir / "feature_scalers.joblib"
)

# Retrieve feature and target information
feature_columns = list(feature_metadata["feature_columns"])
target_column = feature_metadata["target_column"]

# Verify consistency with the loaded datasets
assert len(feature_columns) == 16, "Unexpected number of features."

for year, splits in model_data.items():
    assert year in feature_scalers, f"Missing scaler for {year}"

    for split_name, dataset in splits.items():
        assert dataset["X"].shape[1] == len(feature_columns), (
            f"Feature mismatch: {year} | {split_name}"
        )

print("Feature metadata verification:")
print(f"Number of input features: {len(feature_columns)}")
print(f"Prediction target: {target_column}")

print("\nInput features:")
for index, feature in enumerate(feature_columns, start=1):
    print(f"{index:2d}. {feature}")

print("\nAvailable fitted scalers:", list(feature_scalers.keys()))

print("\nPASS: Feature metadata and datasets are consistent.")

Feature metadata verification:
Number of input features: 16
Prediction target: target_NPWD2721

Input features:
 1. NPWD2721
 2. NPWD2802
 3. NPWD2721_lag_60s
 4. NPWD2802_lag_60s
 5. NPWD2721_lag_120s
 6. NPWD2802_lag_120s
 7. NPWD2721_lag_300s
 8. NPWD2802_lag_300s
 9. NPWD2721_rolling_120s_mean
10. NPWD2721_rolling_120s_std
11. NPWD2802_rolling_120s_mean
12. NPWD2802_rolling_120s_std
13. NPWD2721_rolling_300s_mean
14. NPWD2721_rolling_300s_std
15. NPWD2802_rolling_300s_mean
16. NPWD2802_rolling_300s_std

Available fitted scalers: ['annee1', 'annee2']

PASS: Feature metadata and datasets are consistent.


## 3.3 — Baseline Satellite Power Prediction

### 3.3.1 — Persistence Baseline

A persistence baseline predicts future satellite power using the most recently observed power value.

For the 288-second forecast horizon, the predicted value of `target_NPWD2721` is assumed to equal the current `NPWD2721` measurement.

This baseline requires no model training and provides a reference for evaluating whether machine-learning models improve forecasting accuracy.

Prediction performance is evaluated separately for each observation year using the validation datasets. The test datasets are reserved for final model evaluation.

In [5]:
# Section 3.3.1 — Persistence Baseline

# Identify the current satellite power feature
power_feature_index = feature_columns.index("NPWD2721")

baseline_results = []

for year in ("annee1", "annee2"):

    # Use validation data only
    X_val = model_data[year]["validation"]["X"]
    y_val = model_data[year]["validation"]["y"]

    # Recover the original power values from scaled inputs
    scaler = feature_scalers[year]

    # StandardScaler stores the mean and scale of each feature
    current_power = (
        X_val[:, power_feature_index]
        * scaler.scale_[power_feature_index]
        + scaler.mean_[power_feature_index]
    )

    # Persistence prediction
    y_pred = current_power

    # Calculate regression metrics
    mae = mean_absolute_error(y_val, y_pred)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    r2 = r2_score(y_val, y_pred)

    baseline_results.append({
        "Year": year,
        "Model": "Persistence Baseline",
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

baseline_df = pd.DataFrame(baseline_results)

print("Persistence Baseline — Validation Performance")
display(baseline_df.round(4))

Persistence Baseline — Validation Performance


,Year,Model,MAE,RMSE,R2
0,annee1,Persistence Baseline,0.0205,0.0569,0.7664
1,annee2,Persistence Baseline,0.0214,0.0585,0.7482


### 3.3.2 — Ridge Regression Model

Ridge Regression is used as the first trained machine-learning model for satellite power forecasting.

The model learns relationships between the 16 input features and the future satellite power target.

Separate models are trained for each observation year using the training datasets. Performance is evaluated on the corresponding validation datasets and compared with the persistence baseline.

Ridge Regression uses L2 regularization to reduce model complexity and improve generalization.

In [6]:
# Section 3.3.2 — Train and Evaluate Ridge Regression

ridge_models = {}
ridge_results = []

for year in ("annee1", "annee2"):

    # Retrieve existing model-ready datasets
    X_train = model_data[year]["train"]["X"]
    y_train = model_data[year]["train"]["y"]

    X_val = model_data[year]["validation"]["X"]
    y_val = model_data[year]["validation"]["y"]

    # Initialize and train Ridge Regression
    model = Ridge(alpha=1.0)

    model.fit(X_train, y_train)

    # Predict future satellite power
    y_pred = model.predict(X_val)

    # Evaluate prediction performance
    mae = mean_absolute_error(y_val, y_pred)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    r2 = r2_score(y_val, y_pred)

    # Store trained model and evaluation results
    ridge_models[year] = model

    ridge_results.append({
        "Year": year,
        "Model": "Ridge Regression",
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

ridge_df = pd.DataFrame(ridge_results)

print("Ridge Regression — Validation Performance")
display(ridge_df.round(4))

Ridge Regression — Validation Performance


,Year,Model,MAE,RMSE,R2
0,annee1,Ridge Regression,0.0288,0.0547,0.7838
1,annee2,Ridge Regression,0.0305,0.0562,0.7674


### 3.3.3 — Histogram Gradient Boosting Regression

Histogram Gradient Boosting Regression is introduced to capture nonlinear relationships between current satellite power measurements, lagged values, and rolling statistics.

Separate models are trained for each observation year. To reduce computational cost, a reproducible subset of the chronological training data is used, while validation is performed on the complete validation datasets.

The model is evaluated using MAE, RMSE, and R² to determine whether nonlinear learning improves prediction accuracy compared with persistence and Ridge Regression.

In [7]:
# Section 3.3.3 — Train and Evaluate Histogram Gradient Boosting

hgb_models = {}
hgb_results = []

MAX_TRAIN_SAMPLES = 300_000

for year in ("annee1", "annee2"):

    X_train = model_data[year]["train"]["X"]
    y_train = model_data[year]["train"]["y"]

    X_val = model_data[year]["validation"]["X"]
    y_val = model_data[year]["validation"]["y"]

    # Use evenly spaced training observations to reduce computation
    sample_indices = np.linspace(
        0,
        len(X_train) - 1,
        min(MAX_TRAIN_SAMPLES, len(X_train)),
        dtype=int
    )

    X_sample = X_train[sample_indices]
    y_sample = y_train[sample_indices]

    # Initialize nonlinear model
    model = HistGradientBoostingRegressor(
        max_iter=100,
        max_leaf_nodes=31,
        learning_rate=0.1,
        l2_regularization=1.0,
        random_state=RANDOM_STATE
    )

    print(f"Training nonlinear model for {year}...")

    model.fit(X_sample, y_sample)

    # Predict on the complete validation dataset
    y_pred = model.predict(X_val)

    # Calculate evaluation metrics
    mae = mean_absolute_error(y_val, y_pred)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    r2 = r2_score(y_val, y_pred)

    hgb_models[year] = model

    hgb_results.append({
        "Year": year,
        "Model": "Histogram Gradient Boosting",
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

hgb_df = pd.DataFrame(hgb_results)

print("\nHistogram Gradient Boosting — Validation Performance")
display(hgb_df.round(4))

Training nonlinear model for annee1...
Training nonlinear model for annee2...

Histogram Gradient Boosting — Validation Performance


,Year,Model,MAE,RMSE,R2
0,annee1,Histogram Gradient Boosting,0.0286,0.0546,0.7852
1,annee2,Histogram Gradient Boosting,0.0305,0.0562,0.7676


## 3.4 — Model Comparison and Selection

This section compares the persistence baseline, Ridge Regression, and Histogram Gradient Boosting using their validation performance.

The comparison considers MAE, RMSE, and R². Since the metrics may favor different models, the selection is based on a clearly defined primary metric rather than model complexity alone.

RMSE is used as the primary selection metric because larger forecasting errors may be particularly important when assessing satellite power behavior. MAE and R² are retained as supporting metrics.

The test datasets remain untouched during model selection.

In [8]:
# Section 3.4 — Compare Models and Select Candidates

# Combine previously calculated validation results
comparison_df = pd.concat(
    [baseline_df, ridge_df, hgb_df],
    ignore_index=True
)

# Rank models by validation RMSE within each year
comparison_df = comparison_df.sort_values(
    ["Year", "RMSE", "MAE"],
    ascending=[True, True, True]
).reset_index(drop=True)

comparison_df["RMSE_Rank"] = (
    comparison_df.groupby("Year")["RMSE"]
    .rank(method="min", ascending=True)
    .astype(int)
)

print("Model Comparison — Validation Performance")
display(comparison_df.round(4))

# Identify the best model for each observation year
selected_models_df = (
    comparison_df
    .groupby("Year", as_index=False)
    .first()
    [["Year", "Model", "MAE", "RMSE", "R2"]]
)

print("\nSelected Model Candidates — Lowest Validation RMSE")
display(selected_models_df.round(4))

Model Comparison — Validation Performance


,Year,Model,MAE,RMSE,R2,RMSE_Rank
0,annee1,Histogram Gradient Boosting,0.0286,0.0546,0.7852,1
1,annee1,Ridge Regression,0.0288,0.0547,0.7838,2
2,annee1,Persistence Baseline,0.0205,0.0569,0.7664,3
3,annee2,Histogram Gradient Boosting,0.0305,0.0562,0.7676,1
4,annee2,Ridge Regression,0.0305,0.0562,0.7674,2
5,annee2,Persistence Baseline,0.0214,0.0585,0.7482,3



Selected Model Candidates — Lowest Validation RMSE


,Year,Model,MAE,RMSE,R2
0,annee1,Histogram Gradient Boosting,0.0286,0.0546,0.7852
1,annee2,Histogram Gradient Boosting,0.0305,0.0562,0.7676


## 3.5 — Final Model Evaluation

The selected Histogram Gradient Boosting models are evaluated on the previously unseen test datasets for both observation years.

Model selection was performed using validation RMSE. The test datasets are used only for final performance assessment.

The evaluation reports MAE, RMSE, and R² and compares the selected models against the persistence baseline on the same test observations.

No additional training, preprocessing, or hyperparameter tuning is performed during this stage.

In [10]:
# Section 3.5 — Final Model Evaluation

test_results = []

for year in ("annee1", "annee2"):

    X_test = model_data[year]["test"]["X"]
    y_test = model_data[year]["test"]["y"]

    # Retrieve the model selected during validation
    selected_name = selected_models_df.loc[
        selected_models_df["Year"] == year, "Model"
    ].iloc[0]

    assert selected_name == "Histogram Gradient Boosting"

    model = hgb_models[year]

    # Generate predictions on unseen test data
    y_pred = model.predict(X_test)

    # Evaluate selected model
    test_results.append({
        "Year": year,
        "Model": selected_name,
        "MAE": mean_absolute_error(y_test, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
        "R2": r2_score(y_test, y_pred)
    })

    # Evaluate persistence on the same test observations
    scaler = feature_scalers[year]

    persistence_pred = (
        X_test[:, power_feature_index]
        * scaler.scale_[power_feature_index]
        + scaler.mean_[power_feature_index]
    )

    test_results.append({
        "Year": year,
        "Model": "Persistence Baseline",
        "MAE": mean_absolute_error(y_test, persistence_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, persistence_pred)),
        "R2": r2_score(y_test, persistence_pred)
    })

test_df = pd.DataFrame(test_results)

print("Final Test Performance")
display(test_df.round(4))

Final Test Performance


,Year,Model,MAE,RMSE,R2
0,annee1,Histogram Gradient Boosting,0.0282,0.0547,0.7634
1,annee1,Persistence Baseline,0.0207,0.0573,0.7404
2,annee2,Histogram Gradient Boosting,0.0373,0.0701,0.6841
3,annee2,Persistence Baseline,0.0287,0.0720,0.6670


## 3.6 — Export Trained Prediction Models

The selected Histogram Gradient Boosting models are saved for reuse in the satellite power simulation.

Separate models are retained for `annee1` and `annee2`, consistent with the year-specific preprocessing and feature scaling.

Saving the models eliminates the need to repeat training when running the simulation.

In [11]:
# Section 3.6 — Export Trained Prediction Models

# Store models alongside the existing preprocessing artifacts
model_dir = data_dir / "models"
model_dir.mkdir(parents=True, exist_ok=True)

saved_model_paths = {}

for year in ("annee1", "annee2"):

    model_path = model_dir / f"{year}_hgb_model.joblib"

    # Save the selected trained model
    joblib.dump(hgb_models[year], model_path)

    # Verify that the saved file exists
    assert model_path.is_file()

    saved_model_paths[year] = model_path

    print(f"PASS: {year} model saved to {model_path}")

print("\nAll selected AI models exported successfully.")

PASS: annee1 model saved to C:\Users\DAMISA\Desktop\DAMISA\ECE M2\M2 Sem 1\FABLAB\Dream Orbit\Dream-in-Orbit-an-AI-that-simulates-a-satellite\data\processed\models\annee1_hgb_model.joblib
PASS: annee2 model saved to C:\Users\DAMISA\Desktop\DAMISA\ECE M2\M2 Sem 1\FABLAB\Dream Orbit\Dream-in-Orbit-an-AI-that-simulates-a-satellite\data\processed\models\annee2_hgb_model.joblib

All selected AI models exported successfully.


### 3.6.1 — Verify Saved Prediction Models

This section verifies that the exported machine-learning models can be successfully reloaded and used for satellite power prediction.

Predictions generated by the reloaded models are compared with those from the original trained models to confirm consistency.

A small sample of validation observations is used to minimize computational overhead.

In [12]:
# Section 3.6.1 — Verify Saved Prediction Models

for year in ("annee1", "annee2"):

    # Load the saved model
    loaded_model = joblib.load(saved_model_paths[year])

    # Use a small validation sample
    X_sample = model_data[year]["validation"]["X"][:100]

    # Generate predictions using both models
    original_predictions = hgb_models[year].predict(X_sample)
    loaded_predictions = loaded_model.predict(X_sample)

    # Verify that predictions are identical
    np.testing.assert_allclose(
        original_predictions,
        loaded_predictions,
        rtol=1e-10,
        atol=1e-12
    )

    print(f"PASS: {year} saved model produces consistent predictions.")

print("\nAll saved prediction models verified successfully.")

PASS: annee1 saved model produces consistent predictions.
PASS: annee2 saved model produces consistent predictions.

All saved prediction models verified successfully.


## 3.7 — Modeling Summary and Conclusion

### 3.7.1 — Summary of Model Development

This notebook developed and evaluated machine-learning approaches for predicting satellite electrical power approximately 288 seconds (4.8 minutes) ahead.

The modeling workflow used 16 previously engineered input features from two observation years (`annee1` and `annee2`). The datasets were loaded directly from Notebook 02 without repeating preprocessing or feature scaling.

Three forecasting approaches were evaluated:

1. **Persistence Baseline:** Predicts future satellite power using the current measurement.
2. **Ridge Regression:** Uses regularized linear regression to learn relationships between satellite power features and future measurements.
3. **Histogram Gradient Boosting:** Captures nonlinear relationships between satellite power measurements and historical behavior.

Models were compared using validation MAE, RMSE, and R². RMSE was selected as the primary model-selection metric.

### 3.7.2 — Final Test Performance

| Model | Observation Year | MAE | RMSE | R² |
|---|---|---:|---:|---:|
| Histogram Gradient Boosting | annee1 | 0.0282 | 0.0547 | 0.7634 |
| Persistence Baseline | annee1 | 0.0207 | 0.0573 | 0.7404 |
| Histogram Gradient Boosting | annee2 | 0.0373 | 0.0701 | 0.6841 |
| Persistence Baseline | annee2 | 0.0287 | 0.0720 | 0.6670 |

Histogram Gradient Boosting achieved lower RMSE and higher R² than persistence in both observation years.

However, persistence achieved lower MAE, indicating that the nonlinear model did not outperform the baseline across all evaluation metrics.

### 3.7.3 — Model Selection and Limitations

Histogram Gradient Boosting was selected for both years based on validation RMSE.

The results demonstrate that machine learning can improve short-term satellite power forecasting, although the improvement over persistence is relatively modest.

The models have several limitations:

- Predictions depend on the quality and representativeness of historical satellite measurements.
- Forecasting performance may vary across different operating periods.
- The current models predict one satellite power variable rather than the complete satellite electrical system.
- Model performance has been evaluated using historical data, not live satellite telemetry.

### 3.7.4 — Conclusion and Next Phase

The selected models were successfully exported and verified for consistent predictions after reloading.

The next phase will integrate these trained models into a simplified, data-driven satellite power simulation.

The simulation will use historical satellite measurements to demonstrate how predicted power evolves over time, providing a foundation for interactive visualization and further development of the Dream in Orbit project.